# 01 — Tanore Planet Preparation + NDVI

এই notebook চারটি PlanetScope image থেকে তৈরি করবে:

- 4টি prepared 4-band Surface Reflectance image
- 4টি NDVI image

গুরুত্বপূর্ণ:

- **Common valid mask ব্যবহার করা হবে না**
- প্রতিটি তারিখে তার নিজস্ব valid pixels ব্যবহার হবে
- Light haze রাখা হবে, কিন্তু cloud, shadow, heavy haze, snow ও unusable pixels বাদ যাবে
- পুরোনো invalid output থাকলে একই নামে overwrite হবে
- Final local-PC output থাকবে শুধু দুই folder-এ


### Fixed in this version

UDM2-এর `0` এবং `1` দুটিই বাস্তব mask value। তাই `0`-কে NoData হিসেবে আর ব্যবহার করা হয়নি।  
Reprojection-এর বাইরে শুধু `255` NoData ব্যবহার হবে। ফলে repeated warning আর আসবে না।

## VS Code local-PC version

এই notebook Google Colab বা local PC ব্যবহার করে না।

মূল project path:

```text
D:\Boro Rice Classification
```

Notebook চালানোর আগে `setup_windows.bat` চালিয়ে `Python (Boro Rice Project)` kernel নির্বাচন করুন।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:
# CELL 1 — Portable project path for local use and GitHub reproduction

from pathlib import Path
import os

# Recommended: set BORO_PROJECT_ROOT to the local project directory.
# If it is not set, launch Jupyter from the repository root.
PROJECT_ROOT = Path(
    os.environ.get("BORO_PROJECT_ROOT", str(Path.cwd()))
).expanduser().resolve()

DATA_ROOT = PROJECT_ROOT / "Data"
OUTPUTS_ROOT = PROJECT_ROOT / "Outputs"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Data directory was not found: {DATA_ROOT}\n"
        "Set BORO_PROJECT_ROOT or launch Jupyter from the repository root."
    )

OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Outputs root:", OUTPUTS_ROOT)


In [ ]:
# CELL 2 — Import installed local packages

from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.windows import transform as window_transform
from rasterio.features import geometry_mask
from shapely.geometry import mapping

print("Rasterio:", rasterio.__version__)
print("GeoPandas:", gpd.__version__)

In [ ]:
# CELL 3 — Local Tanore project paths

STUDY_AREA = "Tanore"

RAW_ROOT = (
    DATA_ROOT
    / STUDY_AREA
    / "Planet_Raw"
)

BOUNDARY_DIR = (
    DATA_ROOT
    / STUDY_AREA
    / "Boundary"
)

PREPARED_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Prepared_Planet"
)

NDVI_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Planet_NDVI"
)

REPORT_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Preparation_Reports"
)

for folder in [
    PREPARED_DIR,
    NDVI_DIR,
    REPORT_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

DATE_CONFIG = {
    "20260125": {
        "folder_tokens": [
            "20260125",
            "25_jan",
            "25jan",
        ],
        "label": "25_jan",
    },
    "20260306": {
        "folder_tokens": [
            "20260306",
            "6_mar",
            "6_march",
            "6march",
        ],
        "label": "6_march",
    },
    "20260407": {
        "folder_tokens": [
            "20260407",
            "7_apr",
            "7_april",
            "7april",
        ],
        "label": "7_april",
    },
    "20260422": {
        "folder_tokens": [
            "20260422",
            "22_apr",
            "22_april",
            "22april",
        ],
        "label": "22_april",
    },
}

REFERENCE_DATE = "20260125"

MASK_SNOW = True
MASK_SHADOW = True
MASK_HEAVY_HAZE = True
MASK_CLOUD = True
MASK_UNUSABLE = True

NODATA = -9999.0
BLOCK_SIZE = 512
ALL_TOUCHED = False

print("Raw Planet:", RAW_ROOT)
print("Boundary:", BOUNDARY_DIR)
print("Prepared output:", PREPARED_DIR)
print("NDVI output:", NDVI_DIR)


In [ ]:
# CELL 4 — Helper functions

def normalize_name(text):
    return (
        str(text).lower()
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
    )


def iter_windows(width, height, block_size=512):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=min(block_size, width - col_off),
                height=min(block_size, height - row_off),
            )


def find_raw_folder(tokens):
    if not RAW_ROOT.exists():
        raise FileNotFoundError(f"raw_planet folder পাওয়া যায়নি: {RAW_ROOT}")

    normalized_tokens = [normalize_name(token) for token in tokens]
    matches = []

    for folder in RAW_ROOT.iterdir():
        if not folder.is_dir():
            continue

        normalized_folder = normalize_name(folder.name)

        if any(token in normalized_folder for token in normalized_tokens):
            matches.append(folder)

    if len(matches) != 1:
        raise ValueError(
            f"Raw folder নির্দিষ্ট করা যায়নি। Tokens={tokens}, "
            f"Matches={[str(p) for p in matches]}"
        )

    return matches[0]


def find_sr_and_udm2(folder):
    tiffs = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    )

    udm2_candidates = [
        p for p in tiffs
        if "udm2" in p.name.lower()
    ]

    sr_candidates = [
        p for p in tiffs
        if "udm2" not in p.name.lower()
        and "ndvi" not in p.name.lower()
    ]

    if len(udm2_candidates) != 1:
        raise ValueError(
            f"{folder.name}: exactly one UDM2 TIFF দরকার। "
            f"Found={[p.name for p in udm2_candidates]}"
        )

    if len(sr_candidates) == 0:
        raise FileNotFoundError(
            f"{folder.name}: main Planet TIFF পাওয়া যায়নি।"
        )

    composite = folder / "composite.tif"

    if composite.exists():
        sr_path = composite
    elif len(sr_candidates) == 1:
        sr_path = sr_candidates[0]
    else:
        # Prefer the largest non-UDM2 TIFF.
        sr_path = max(
            sr_candidates,
            key=lambda path: path.stat().st_size,
        )

    return sr_path, udm2_candidates[0]


def find_tanore_boundary():
    excluded = {
        "rice",
        "nonrice",
        "train",
        "training",
        "validation",
        "sample",
        "point",
    }

    candidates = []

    for shp in BOUNDARY_DIR.rglob("*.shp"):
        name = shp.stem.lower()

        if "tanore" not in name:
            continue

        if any(word in name for word in excluded):
            continue

        try:
            layer = gpd.read_file(shp)

            if layer.empty or layer.crs is None:
                continue

            geom_types = set(
                layer.geometry.dropna().geom_type.str.lower()
            )

            if not geom_types.intersection({"polygon", "multipolygon"}):
                continue

            if layer.crs.is_geographic:
                area = float(
                    layer.to_crs(32645).geometry.area.sum()
                )
            else:
                area = float(layer.geometry.area.sum())

            candidates.append((area, shp))

        except Exception:
            continue

    if not candidates:
        raise FileNotFoundError(
            "Tanore polygon boundary shapefile পাওয়া যায়নি। "
            "Boundary .shp/.shx/.dbf/.prj একসঙ্গে "
            f"{BOUNDARY_DIR}-এর ভিতরে রাখুন।"
        )

    candidates.sort(reverse=True, key=lambda item: item[0])
    return candidates[0][1]


def infer_band_map(count):
    if count == 4:
        return [1, 2, 3, 4]

    if count == 8:
        # Comparable Blue, Green, Red, NIR for SuperDove.
        return [2, 4, 6, 8]

    raise ValueError(
        f"Planet image has {count} bands; expected 4 or 8."
    )


def detect_scale(src):
    # Raw Planet SR is normally 0–10000.
    # Scan blocks until valid values are found.
    nodata = src.nodata

    for _, window in src.block_windows(1):
        data = src.read(
            window=window,
            masked=False,
        ).astype("float32")

        valid = np.isfinite(data)

        if nodata is not None:
            valid &= data != nodata

        valid &= data > 0
        values = data[valid]

        if values.size:
            median_value = float(np.median(values))
            return 0.0001 if median_value > 2.0 else 1.0

    raise ValueError(f"No valid raw Planet values found: {src.name}")


def quality_valid_from_udm2(udm):
    if udm.shape[0] < 8:
        raise ValueError(
            f"UDM2 has {udm.shape[0]} bands; expected at least 8."
        )

    # 0 and 1 are valid UDM2 values.
    # 255 is used only outside the source footprint after reprojection.
    coverage_valid = np.all(udm != 255, axis=0)

    snow = udm[1] > 0
    shadow = udm[2] > 0
    heavy_haze = udm[4] > 0
    cloud = udm[5] > 0
    unusable = udm[7] > 0

    valid = coverage_valid.copy()

    if MASK_SNOW:
        valid &= ~snow
    if MASK_SHADOW:
        valid &= ~shadow
    if MASK_HEAVY_HAZE:
        valid &= ~heavy_haze
    if MASK_CLOUD:
        valid &= ~cloud
    if MASK_UNUSABLE:
        valid &= ~unusable

    return valid


def clip_window_from_bounds(bounds, transform, width, height):
    minx, miny, maxx, maxy = bounds

    col_a, row_a = (~transform) * (minx, maxy)
    col_b, row_b = (~transform) * (maxx, miny)

    col0 = max(0, int(np.floor(min(col_a, col_b))))
    row0 = max(0, int(np.floor(min(row_a, row_b))))
    col1 = min(width, int(np.ceil(max(col_a, col_b))))
    row1 = min(height, int(np.ceil(max(row_a, row_b))))

    if col1 <= col0 or row1 <= row0:
        raise ValueError("Boundary does not overlap the January Planet image.")

    return Window(col0, row0, col1 - col0, row1 - row0)


def output_profile(grid, count):
    return {
        "driver": "GTiff",
        "width": grid["width"],
        "height": grid["height"],
        "count": count,
        "dtype": "float32",
        "crs": grid["crs"],
        "transform": grid["transform"],
        "nodata": NODATA,
        "compress": "DEFLATE",
        "predictor": 3,
        "tiled": True,
        "blockxsize": BLOCK_SIZE,
        "blockysize": BLOCK_SIZE,
        "BIGTIFF": "IF_SAFER",
    }


In [ ]:
# CELL 5 — Resolve inputs and create one common Planet grid

resolved = {}

for date, config in DATE_CONFIG.items():
    folder = find_raw_folder(config["folder_tokens"])
    sr_path, udm2_path = find_sr_and_udm2(folder)

    resolved[date] = {
        **config,
        "folder": folder,
        "sr": sr_path,
        "udm2": udm2_path,
    }

    print(
        date,
        "→",
        folder.name,
        "| SR:",
        sr_path.name,
        "| UDM2:",
        udm2_path.name,
    )

AOI_PATH = find_tanore_boundary()
print("Boundary:", AOI_PATH)

aoi = gpd.read_file(AOI_PATH)

aoi = aoi[
    aoi.geometry.notna()
    & ~aoi.geometry.is_empty
].copy()

try:
    from shapely import make_valid
    aoi["geometry"] = aoi.geometry.apply(make_valid)
except Exception:
    aoi["geometry"] = aoi.geometry.buffer(0)

aoi = aoi[
    aoi.geometry.notna()
    & ~aoi.geometry.is_empty
].copy()

aoi = aoi.dissolve().reset_index(drop=True)

reference_path = resolved[REFERENCE_DATE]["sr"]

with rasterio.open(reference_path) as reference:
    aoi_ref = aoi.to_crs(reference.crs)
    geometry = aoi_ref.geometry.iloc[0]

    clip_window = clip_window_from_bounds(
        geometry.bounds,
        reference.transform,
        reference.width,
        reference.height,
    )

    clip_transform = window_transform(
        clip_window,
        reference.transform,
    )

    GRID = {
        "crs": reference.crs,
        "transform": clip_transform,
        "width": int(clip_window.width),
        "height": int(clip_window.height),
    }

    AOI_MASK = geometry_mask(
        [mapping(geometry)],
        out_shape=(GRID["height"], GRID["width"]),
        transform=GRID["transform"],
        invert=True,
        all_touched=ALL_TOUCHED,
    )

print("Grid size:", GRID["width"], "x", GRID["height"])
print("AOI pixels:", int(AOI_MASK.sum()))


In [ ]:
# CELL 6 — Create prepared 4-band images and NDVI

report_rows = []

for date, item in resolved.items():
    label = item["label"]

    prepared_path = (
        PREPARED_DIR
        / f"Tanore_{label}_SR_prepared_clip.tif"
    )

    ndvi_path = (
        NDVI_DIR
        / f"Tanore_{label}_NDVI_clip.tif"
    )

    valid_pixel_count = 0
    ndvi_sum = 0.0
    ndvi_min = np.inf
    ndvi_max = -np.inf

    with rasterio.open(item["sr"]) as sr_src:
        with rasterio.open(item["udm2"]) as udm_src:
            selected_bands = infer_band_map(sr_src.count)
            scale_factor = detect_scale(sr_src)

            sr_nodata = (
                sr_src.nodata
                if sr_src.nodata is not None
                else 0
            )

            with WarpedVRT(
                sr_src,
                crs=GRID["crs"],
                transform=GRID["transform"],
                width=GRID["width"],
                height=GRID["height"],
                resampling=Resampling.bilinear,
                src_nodata=sr_nodata,
                nodata=NODATA,
                dtype="float32",
            ) as sr_vrt:
                with WarpedVRT(
                    udm_src,
                    crs=GRID["crs"],
                    transform=GRID["transform"],
                    width=GRID["width"],
                    height=GRID["height"],
                    resampling=Resampling.nearest,
                    # UDM2 uses 0/1 as real data. Never use 0 as NoData.
                    src_nodata=255,
                    nodata=255,
                    dtype="uint8",
                ) as udm_vrt:
                    with rasterio.open(
                        prepared_path,
                        "w",
                        **output_profile(GRID, 4),
                    ) as prepared_dst:
                        with rasterio.open(
                            ndvi_path,
                            "w",
                            **output_profile(GRID, 1),
                        ) as ndvi_dst:
                            descriptions = [
                                "Blue_Surface_Reflectance",
                                "Green_Surface_Reflectance",
                                "Red_Surface_Reflectance",
                                "NIR_Surface_Reflectance",
                            ]

                            for index, description in enumerate(
                                descriptions,
                                start=1,
                            ):
                                prepared_dst.set_band_description(
                                    index,
                                    description,
                                )

                            ndvi_dst.set_band_description(1, "NDVI")

                            prepared_dst.update_tags(
                                study_area=STUDY_AREA,
                                acquisition_date=date,
                                source=str(item["sr"]),
                                source_udm2=str(item["udm2"]),
                                scale_factor=scale_factor,
                                mask_method=(
                                    "Per-date AOI + UDM2; no common mask; "
                                    "light haze retained"
                                ),
                            )

                            ndvi_dst.update_tags(
                                study_area=STUDY_AREA,
                                acquisition_date=date,
                                source_prepared=str(prepared_path),
                                formula="(NIR-Red)/(NIR+Red)",
                            )

                            for window in iter_windows(
                                GRID["width"],
                                GRID["height"],
                                BLOCK_SIZE,
                            ):
                                row0 = int(window.row_off)
                                row1 = row0 + int(window.height)
                                col0 = int(window.col_off)
                                col1 = col0 + int(window.width)

                                aoi_block = AOI_MASK[
                                    row0:row1,
                                    col0:col1,
                                ]

                                raw = sr_vrt.read(
                                    selected_bands,
                                    window=window,
                                ).astype("float32")

                                udm = udm_vrt.read(window=window)

                                reflectance = raw * scale_factor

                                quality_valid = quality_valid_from_udm2(
                                    udm
                                )

                                spectral_valid = (
                                    np.all(
                                        np.isfinite(reflectance),
                                        axis=0,
                                    )
                                    & np.all(raw != NODATA, axis=0)
                                    & np.any(raw > 0, axis=0)
                                )

                                valid = (
                                    aoi_block
                                    & quality_valid
                                    & spectral_valid
                                )

                                prepared = np.full(
                                    reflectance.shape,
                                    NODATA,
                                    dtype="float32",
                                )

                                prepared[:, valid] = (
                                    reflectance[:, valid]
                                )

                                prepared_dst.write(
                                    prepared,
                                    window=window,
                                )

                                red = reflectance[2]
                                nir = reflectance[3]
                                denominator = nir + red

                                ndvi_valid = (
                                    valid
                                    & np.isfinite(denominator)
                                    & (np.abs(denominator) > 1e-8)
                                )

                                ndvi = np.full(
                                    red.shape,
                                    NODATA,
                                    dtype="float32",
                                )

                                values = (
                                    (
                                        nir[ndvi_valid]
                                        - red[ndvi_valid]
                                    )
                                    / denominator[ndvi_valid]
                                )

                                values = np.clip(values, -1.0, 1.0)
                                ndvi[ndvi_valid] = values

                                ndvi_dst.write(
                                    ndvi,
                                    1,
                                    window=window,
                                )

                                valid_pixel_count += int(
                                    ndvi_valid.sum()
                                )

                                if values.size:
                                    ndvi_sum += float(
                                        values.sum(dtype="float64")
                                    )
                                    ndvi_min = min(
                                        ndvi_min,
                                        float(values.min()),
                                    )
                                    ndvi_max = max(
                                        ndvi_max,
                                        float(values.max()),
                                    )

    if valid_pixel_count == 0:
        raise ValueError(
            f"{date}: zero valid pixels. Check raw SR/UDM2 files."
        )

    report_rows.append({
        "date": date,
        "prepared_path": str(prepared_path),
        "ndvi_path": str(ndvi_path),
        "valid_pixels": valid_pixel_count,
        "valid_percent_of_AOI": (
            100.0
            * valid_pixel_count
            / int(AOI_MASK.sum())
        ),
        "ndvi_min": ndvi_min,
        "ndvi_max": ndvi_max,
        "ndvi_mean": ndvi_sum / valid_pixel_count,
        "scale_factor": scale_factor,
    })

    print(
        f"✅ {date}: prepared image + NDVI complete | "
        f"valid={valid_pixel_count:,}"
    )

preparation_report = pd.DataFrame(report_rows)
display(preparation_report)


In [ ]:
# CELL 7 — Final verification

verification_rows = []

for date, config in DATE_CONFIG.items():
    label = config["label"]

    prepared_path = (
        PREPARED_DIR
        / f"Tanore_{label}_SR_prepared_clip.tif"
    )

    ndvi_path = (
        NDVI_DIR
        / f"Tanore_{label}_NDVI_clip.tif"
    )

    with rasterio.open(prepared_path) as prepared:
        with rasterio.open(ndvi_path) as ndvi:
            if prepared.count != 4:
                raise ValueError(
                    f"{prepared_path.name}: expected 4 bands."
                )

            if ndvi.count != 1:
                raise ValueError(
                    f"{ndvi_path.name}: expected 1 band."
                )

            same_grid = (
                prepared.crs == ndvi.crs
                and prepared.transform == ndvi.transform
                and prepared.width == ndvi.width
                and prepared.height == ndvi.height
            )

            if not same_grid:
                raise ValueError(
                    f"{date}: prepared and NDVI grids differ."
                )

            verification_rows.append({
                "date": date,
                "prepared": prepared_path.name,
                "ndvi": ndvi_path.name,
                "bands_prepared": prepared.count,
                "bands_ndvi": ndvi.count,
                "width": prepared.width,
                "height": prepared.height,
                "crs": str(prepared.crs),
                "grid_match": same_grid,
            })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)

print("✅ PREPARATION COMPLETE")
print("Prepared images:", PREPARED_DIR)
print("NDVI images:", NDVI_DIR)
